Code to set path root

In [ ]:
import sys
import os
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


sys.path.append(os.path.abspath(".."))

# Training model on `fight-weaponized-other-dataset` with 64x64 Image Sizes
* ResNet Included
* using `datasets`, `transforms` module from `torchvison`
* using `dataloader` module from `torch.utils.data`

## Importing necessary Modules

In [ ]:
# Import torch libraries
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn

# Import modules
from modules.architectures.Architecture import Architecture, ResidualBlock, Shortcut
from modules.helper.Trainer import Trainer
from modules.helper.Plotter import plot_training_metrics, plot_testing_history
from modules.helper.Tester import  Tester

Check if CUDA is used

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))
    print("Current device index:", torch.cuda.current_device())
    print("Device count:", torch.cuda.device_count())
else:
    print("Running on CPU")

### Use datasets, dataloader and transforms for loading training Dataset

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])
train_dataset = datasets.ImageFolder(
    root = "../datasets/fight-weaponized-other-dataset/train",
    transform = train_transform
)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=16,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    shuffle=True
)

print("Total Batches => ", len(train_dataloader))

### Use datasets, dataloader and transforms for loading validation Dataset

In [ ]:
val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

val_dataset = datasets.ImageFolder(
    root = "../datasets/fight-weaponized-other-dataset/val",
    transform = val_transform
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=16,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Total Batches => ", len(val_dataloader))

### Using Model Architecture:
* 10 Convolutional Layers
    - Conv2D
    - BatchNorm2D
    - ReLu
    - MaxPool2D (Optional)
* 1 Linear Layer
* SDG Optimizer

In [ ]:
model = Architecture().to("cuda")

### Adding 100 blocks (MaxPool2D in each 20 block)

In [ ]:
out_channels = 8
size = 64

model_blocks = [
    nn.Conv2d(3, out_channels, 3, 1, 1),
    nn.BatchNorm2d(out_channels),
    nn.ReLU()
]

for stage in range(5):

    for i in range(20):

        conv = nn.Conv2d(out_channels, out_channels, 3, 1, 1)
        bn = nn.BatchNorm2d(out_channels)

        model_blocks.append(
            ResidualBlock([conv, bn, nn.ReLU()])
        )

    if stage < 3:
        model_blocks.append(nn.MaxPool2d(2, 2))
        size //= 2

    if stage < 4:
        new_channels = out_channels * 2

        model_blocks.append(
            ResidualBlock(
                [
                    nn.Conv2d(out_channels, new_channels, 3, 1, 1),
                    nn.BatchNorm2d(new_channels),
                    nn.ReLU()
                ],
                Shortcut(out_channels, new_channels)
            )
        )

        out_channels = new_channels


print(f"Final Out Channels = {out_channels}")
print(f"Final Shape = {size}")

In [ ]:
model = model.add(
    # Conv Blocks
    *model_blocks,
    
    # Flatten
    nn.Flatten(),

    nn.Linear(out_channels * size * size, 128),
    nn.ReLU(),
    nn.Linear(128, 3)
    )

### Use Trainer to train and check validations
Adding weight decay and decreased weight

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=3e-4, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:
trainer = Trainer(
    model, 
    train_dataloader, 
    val_dataloader, 
    optimizer=optimizer, 
    num_classes=3,
    criterion=criterion,
    device="cuda",
    save_dir="../models/experiment8/",
    save_checkpoints=10,
    print_every=5
    )

In [ ]:
history = trainer.fit(100)

### Save Metrics

In [ ]:
df = plot_training_metrics(history)
df.to_csv("../documentations/experiments/experiment8/tables/training_metrics.csv", index=False)

### Training/Validation Trend (100 epochs)
* Training loss shows a consistent overall downward trend from ~1.32 to ~0.03–0.06 range, indicating effective optimization.
* Training accuracy steadily increases, moving from ~0.35 to peaks above 0.99, showing strong memorization capacity.
* Validation accuracy improves initially, becomes stable in mid-epochs, and fluctuates in later stages.
* Validation loss is highly unstable, with intermittent sharp spikes (notably around epochs 40, 83, 89, 95–96), indicating periodic overfitting or instability.
* Clear overfitting appears after ~80 epochs where train metrics remain near-perfect but validation degrades.
* Best generalization occurs in mid-to-late training range (roughly epochs 58–84).
* Model shows repeated recovery after instability spikes, suggesting non-convergent validation behavior.
* Precision/recall/F1 on training set become saturated (>0.98) in later epochs, indicating ceiling effect.
* Validation metrics plateau around ~0.74–0.77 F1 at best, showing limited generalization gap closure.
* Performance becomes unstable after prolonged training, with diminishing returns beyond ~70 epochs.

The model learns effectively, as shown by the steady reduction in training loss and rapid improvement in training accuracy, eventually reaching near-perfect performance. However, validation metrics improve only up to a point and then plateau, with repeated instability and sharp degradation episodes, indicating overfitting and sensitivity to training dynamics. The best generalization is achieved in the mid-to-late training phase, after which continued training primarily improves training fit rather than real-world performance. The widening gap between training and validation performance in later epochs confirms that the model begins to over-specialize, and optimal performance occurs before full convergence of training metrics.

<b>Best Epoch 84</b>

<b>Loss</b>
* Train Loss = 0.11835905877414643
* Valid Loss = 1.1191416830426595

<b>Training Metrics</b>
* Train Accuracy = 0.9617202281951904
* Train Precison = 0.9619541168212891
* Train Recall = 0.9618192315101624
* Train F1 = 0.9618772864341736

<b>Validation Accuracy</b>
* Validation Accuracy = 0.7743362784385681
* Validation Precision = 0.7758584022521973
* Validation Recall = 0.7756655812263489
* Validation F1 = 0.7757452130317688

## Use Tester Module to Test Model

Load Model with State Dict

In [ ]:
# Transforms of Data
test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# Dataset Loading From Image dir
test_dataset = datasets.ImageFolder(
    root="../datasets/fight-weaponized-other-dataset/test", 
    transform = test_transform 
    )

# DataLoader
test_loader = DataLoader(
    dataset=test_dataset, 
    batch_size=16,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
    )

tester = Tester(
    model,
    test_loader,
    3,
    torch.nn.CrossEntropyLoss(),
    "cuda"
)

test_scores = tester.test_all_checkpoints(
    "../models/experiment8"
)

### Save Test Metrics

In [ ]:
# Plot all 100 epochs
test_metrics_df = plot_testing_history(test_scores)
test_metrics_df.to_csv("../documentations/experiments/experiment8/tables/test_metrics.csv", index=False)

### Test Performance Trend
* The model started with low performance at epoch 1, achieving 39.91% test accuracy and 29.74% test F1 score, indicating limited initial feature learning.
* Test performance improved rapidly during early training, reaching 62.94% accuracy by epoch 8 and 70.39% accuracy by epoch 14.
* Test loss generally decreased during the early and middle training stages, showing improved prediction confidence.
* The model crossed 75% test accuracy around epoch 33 and continued improving afterward.
* The model achieved a strong improvement between epochs 33 and 60, increasing test accuracy from 76.32% to 80.92%.
* The model experienced major instability around epochs 40 and 50, where test loss increased sharply and accuracy dropped significantly, suggesting unstable optimization or temporary prediction collapse.
* After epoch 54, the model recovered and maintained consistently strong test performance above 79% accuracy.
* The highest stable performance region occurred between epochs 58 and 80, where test accuracy remained around 81%.
* Test precision remained consistently high throughout training, often exceeding test accuracy, indicating reliable positive predictions.
* Test recall closely followed accuracy in later epochs, showing balanced prediction across classes.
* The model achieved its strongest generalization at epoch 73, reaching the highest test accuracy and near-highest F1 score.
* After epoch 73, performance remained stable but did not improve significantly, suggesting the model reached convergence.
* Epoch 100 showed reduced performance compared with the best epoch, indicating possible overfitting or reduced generalization in the final stage.
* The overall test trend shows successful learning, stable convergence, and good generalization after overcoming early training instability.

The model showed strong learning progression throughout training, improving from poor initial performance to over 81% test accuracy. The largest improvements occurred during the first 60 epochs, followed by a stable high-performance phase. Although the model experienced severe instability at epochs 40 and 50, it recovered and maintained strong test metrics afterward. The best generalization performance was achieved around epoch 73, after which additional training did not provide meaningful improvements and slightly reduced final test performance.
<b>Best Epoch 73</b>

* Loss = 0.6478886659546145
* Accuracy = 0.8135964870452881
* Precision = 0.8167217373847961
* Recall = 0.8146975040435791
* F1-Score = 0.8147904276847839